# ReMask component experiments
Select a GPU runtime; A100/High-RAM is preferable. Add the Colab secret **HF_TOKEN** with read access to the private SUIM dataset. Run the cells in order. Source files are downloaded automatically; target training masks never enter adaptation.

Outputs and the code revision are saved to Drive. Repeat these cells on a replacement runtime to resume. The full experiment matrix requires multiple sessions. Smoke scores are correctness diagnostics, not paper results.


In [ ]:
from pathlib import Path
import os, subprocess, sys, time, venv
from google.colab import drive, userdata

SESSION_STARTED = time.monotonic()
OUTPUT = Path('/content/drive/MyDrive/ReMask_component_experiments_v1')
SUITE = 'priority'  # priority, core, or all
PROTOCOL = 'all'   # or uiis_to_suim / usis10k_to_suim
SEED = 1
DO_FROZEN = True

drive.mount('/content/drive')
OUTPUT.mkdir(parents=True, exist_ok=True)
REPO = Path('/content/ReMask')
if not (REPO / '.git').exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/Kuan2vn/ReMask.git', str(REPO)])
pin = OUTPUT / 'CODE_REVISION.txt'
revision = pin.read_text().strip() if pin.exists() else 'origin/main'
subprocess.check_call(['git', '-C', str(REPO), 'fetch', 'origin'])
subprocess.check_call(['git', '-C', str(REPO), 'checkout', '--detach', revision])
commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if not pin.exists():
    pin.write_text(commit + '\n')
print('Code revision:', commit)

ENV = Path('/content/remask-runtime')
PYTHON = ENV / 'bin/python'
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=False).create(ENV)
subprocess.check_call([sys.executable, '-m', 'pip', '--python', str(PYTHON), 'install', '--upgrade', 'pip'])
subprocess.check_call([str(PYTHON), '-m', 'pip', 'install', 'torch==2.8.0', 'torchvision==0.23.0', '--index-url', 'https://download.pytorch.org/whl/cu126'])
subprocess.check_call([str(PYTHON), '-m', 'pip', 'install', '-r', str(REPO / 'requirements.txt')])
subprocess.check_call([str(PYTHON), '-m', 'pip', 'check'])
subprocess.check_call(['nvidia-smi'])

TOKEN = userdata.get('HF_TOKEN')
DOWNLOAD_ENV = dict(os.environ, HF_TOKEN=TOKEN)
prepare = r"""
from pathlib import Path
import hashlib, zipfile
from huggingface_hub import hf_hub_download
archive = hf_hub_download('kuan2/SUIM-ReMask', 'SUIM.zip', repo_type='dataset', revision='b2997099ded641a1de08d00072bfb268cb9ef6dc')
if hashlib.sha256(Path(archive).read_bytes()).hexdigest() != '86943900de1a74feaf826e0f2e8aa1e2067339c70a6755f91b8bb30522c874cd':
    raise RuntimeError('SUIM archive checksum mismatch')
with zipfile.ZipFile(archive) as bundle:
    bundle.extractall('/content/ReMask/data')
print('Exact SUIM archive verified and extracted')
"""
subprocess.check_call([str(PYTHON), '-c', prepare], env=DOWNLOAD_ENV)
del TOKEN, DOWNLOAD_ENV
DATA = REPO / 'data/SUIM'
def run_script(script, *args):
    subprocess.check_call([str(PYTHON), str(REPO / script), *map(str, args)], cwd=REPO)
run_script('experiments/rebuttal/run.py', '--check', '--protocol', PROTOCOL, '--data', DATA)


## Correctness check
This runs the native model on a small isolated dataset, pauses, and restores a full checkpoint in a new process. It does not establish full-resolution GPU capacity. Full runs retain batch4 and use residual activation checkpointing; they stop with an error if the assigned GPU cannot fit the recipe.


In [ ]:
run_script('experiments/rebuttal/run.py', '--arm', 'F', '--protocol', 'uiis_to_suim',
           '--data', DATA, '--output', OUTPUT, '--smoke', '--smoke-refresh')


## Frozen probe comparison
All three severities, both probe families, pixel/mean/median aggregation, and identity/confidence controls are retained. Completed images are reused after interruption. This is a fixed source-model scoring study, separate from downstream training.


In [ ]:
if DO_FROZEN:
    run_script('experiments/rebuttal/analyze.py', '--protocol', PROTOCOL, '--data', DATA, '--output', OUTPUT)


## Training queue
`priority` starts with the most direct contrasts; `core` covers the main requested experiments; `all` also runs secondary controls. Nine hours is a soft total budget for this notebook session, leaving headroom for checkpointing. A pause preserves progress; repeat the notebook with the same OUTPUT and code pin. Do not run two notebooks against the same OUTPUT simultaneously.


In [ ]:
remaining_hours = 9 - (time.monotonic() - SESSION_STARTED) / 3600
if remaining_hours <= 0:
    print('Setup/analysis used the session budget. Reconnect and rerun to continue.')
else:
    run_script('experiments/rebuttal/run.py', '--suite', SUITE, '--protocol', PROTOCOL,
               '--data', DATA, '--output', OUTPUT, '--seed', SEED,
               '--session-hours', remaining_hours)
run_script('experiments/rebuttal/report.py', '--output', OUTPUT)
print('Reports and checkpoints:', OUTPUT)


## Results
Read `full/results.csv`, `full/quality_over_time.csv`, `full/mismatch_5k_windows.csv`, and the frozen summaries in Drive. A session pause or a completed smoke is not a completed30k experiment. Results include every available arm and both endpoint and selected-peak metrics.
